Setup Imports

In [24]:
# Clear all variables and start fresh
%reset -f
print("✓ Kernel reset, starting fresh")

✓ Kernel reset, starting fresh


In [25]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np

In [26]:
# Force reload of modules to pick up latest changes
import importlib
import sys

# Remove cached imports
for module in list(sys.modules.keys()):
    if 'data_loader' in module or 'features' in module or 'model' in module or 'train' in module or 'explain' in module:
        del sys.modules[module]

print("✓ Modules cleared for reload")

✓ Modules cleared for reload


Import Your Modules

In [27]:
from data_loader import load_data
from features import prepare_molecule_data, encode_labels
from model import get_model
from train import train_model
from explain import explain_model

Load Dataset

In [28]:
molecules, entities, mapping = load_data()

print("Molecules shape:", molecules.shape)
print("Entities shape:", entities.shape)
print("Mapping shape:", mapping.shape)

molecules.head()

Molecules shape: (25595, 43)
Entities shape: (936, 10)
Mapping shape: (60263, 3)


,pubchem_id,iupac_name,common_name,smile,molecular_weight,hbd_count,hba_count,num_rotatablebonds,complexity,topological_polor_surfacearea,...,bitter,supersweetdb_id,bitterdb_id,fooddb_id,flavornet_id,fenoroli_and_os,natural,unknown_natural,synthetic,flavor_profile
0,4,1-aminopropan-2-ol,1-Aminopropan-2-ol,CC(CN)O,75.111,2,2,1,22.9,46.2,...,0,NaN,NaN,FDB008936,0,1,1,0,0,fishy
1,40,"6-(hydroxymethyl)oxane-2,4,5-triol",2-Deoxyhexopyranose,C1C(C(C(OC1O)CO)O)O,164.157,4,5,1,128.0,90.2,...,0,9189.0,NaN,NaN,0,0,0,0,0,sweet
2,47,3-methyl-2-oxopentanoic acid,3-Methyl-2-oxovaleric acid,CCC(C)C(=O)C(=O)O,130.143,1,3,3,128.0,54.4,...,0,NaN,NaN,NaN,0,1,0,0,0,NaN
3,49,3-methyl-2-oxobutanoic acid,3-Methyl-2-oxobutanoic acid,CC(C)C(=O)C(=O)O,116.116,1,3,2,115.0,54.4,...,0,NaN,NaN,FDB012250,0,1,1,0,0,fruity
4,51,2-oxopentanedioic acid,2-ketoglutaric acid,C(CC(=O)O)C(=O)C(=O)O,146.098,2,5,4,171.0,91.7,...,0,NaN,NaN,FDB003361,0,1,0,0,0,odorless


Inspect Data

In [14]:
molecules.columns

Index(['pubchem_id', 'iupac_name', 'common_name', 'smile', 'molecular_weight',
       'hbd_count', 'hba_count', 'num_rotatablebonds', 'complexity',
       'topological_polor_surfacearea', 'monoisotopic_mass', 'exact_mass',
       'xlogp', 'charge', 'heavy_atom_count', 'atom_stereo_count',
       'defined_atom_stereocenter_count', 'undefined_atom_stereocenter_count',
       'bond_stereo_count', 'defined_bond_stereocenter_count',
       'undefined_bond_stereocenter_count', 'isotope_atom_count',
       'covalently_bonded_unit_count', 'cas_id', 'fema_number',
       'fema_flavor_profile', 'odor', 'taste', 'functional_groups', 'inchi',
       'volume3d', 'fooddb_flavor_profile', 'super_sweet', 'bitter',
       'supersweetdb_id', 'bitterdb_id', 'fooddb_id', 'flavornet_id',
       'fenoroli_and_os', 'natural', 'unknown_natural', 'synthetic',
       'flavor_profile'],
      dtype='str')

In [15]:
print("All columns:", molecules.columns.tolist())
print("\nDataframe info:")
molecules.info()
print("\nMissing values per column:")
print(molecules.isnull().sum())
print("\nFirst few rows of molecules:")
print(molecules.head(10))

All columns: ['pubchem_id', 'iupac_name', 'common_name', 'smile', 'molecular_weight', 'hbd_count', 'hba_count', 'num_rotatablebonds', 'complexity', 'topological_polor_surfacearea', 'monoisotopic_mass', 'exact_mass', 'xlogp', 'charge', 'heavy_atom_count', 'atom_stereo_count', 'defined_atom_stereocenter_count', 'undefined_atom_stereocenter_count', 'bond_stereo_count', 'defined_bond_stereocenter_count', 'undefined_bond_stereocenter_count', 'isotope_atom_count', 'covalently_bonded_unit_count', 'cas_id', 'fema_number', 'fema_flavor_profile', 'odor', 'taste', 'functional_groups', 'inchi', 'volume3d', 'fooddb_flavor_profile', 'super_sweet', 'bitter', 'supersweetdb_id', 'bitterdb_id', 'fooddb_id', 'flavornet_id', 'fenoroli_and_os', 'natural', 'unknown_natural', 'synthetic', 'flavor_profile']

Dataframe info:
<class 'pandas.DataFrame'>
RangeIndex: 25595 entries, 0 to 25594
Data columns (total 43 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------               

Prepare Data

In [29]:
df = prepare_molecule_data(molecules)

df.head()

,pubchem_id,iupac_name,common_name,smile,molecular_weight,hbd_count,hba_count,num_rotatablebonds,complexity,topological_polor_surfacearea,...,bitter,supersweetdb_id,bitterdb_id,fooddb_id,flavornet_id,fenoroli_and_os,natural,unknown_natural,synthetic,flavor_profile
0,4,1-aminopropan-2-ol,1-Aminopropan-2-ol,CC(CN)O,75.111,2,2,1,22.9,46.2,...,0,0.0,0.0,FDB008936,0,1,1,0,0,fishy
1,40,"6-(hydroxymethyl)oxane-2,4,5-triol",2-Deoxyhexopyranose,C1C(C(C(OC1O)CO)O)O,164.157,4,5,1,128.0,90.2,...,0,9189.0,0.0,NaN,0,0,0,0,0,sweet
3,49,3-methyl-2-oxobutanoic acid,3-Methyl-2-oxobutanoic acid,CC(C)C(=O)C(=O)O,116.116,1,3,2,115.0,54.4,...,0,0.0,0.0,FDB012250,0,1,1,0,0,fruity
4,51,2-oxopentanedioic acid,2-ketoglutaric acid,C(CC(=O)O)C(=O)C(=O)O,146.098,2,5,4,171.0,91.7,...,0,0.0,0.0,FDB003361,0,1,0,0,0,odorless
5,58,2-oxobutanoic acid,2-oxobutanoic acid,CCC(=O)C(=O)O,102.089,1,3,2,95.1,54.4,...,0,0.0,0.0,FDB003359,0,1,1,0,0,brown@caramel@lactonic@sweet@creamy


In [30]:
print("Shape after prepare_molecule_data:", df.shape)
print("Is it empty?", len(df) == 0)
if len(df) == 0:
    print("\n⚠️  DataFrame is EMPTY after prepare_molecule_data()")
    print("This likely means:")
    print("  - dropna() removed all rows, OR")
    print("  - 'flavor_profile' column doesn't exist, OR")
    print("  - All 'flavor_profile' values are NaN")
else:
    print("\nColumns in processed df:")
    print(df.columns.tolist())
    print("\nDataframe head:")
    print(df.head())

Shape after prepare_molecule_data: (25106, 43)
Is it empty? False

Columns in processed df:
['pubchem_id', 'iupac_name', 'common_name', 'smile', 'molecular_weight', 'hbd_count', 'hba_count', 'num_rotatablebonds', 'complexity', 'topological_polor_surfacearea', 'monoisotopic_mass', 'exact_mass', 'xlogp', 'charge', 'heavy_atom_count', 'atom_stereo_count', 'defined_atom_stereocenter_count', 'undefined_atom_stereocenter_count', 'bond_stereo_count', 'defined_bond_stereocenter_count', 'undefined_bond_stereocenter_count', 'isotope_atom_count', 'covalently_bonded_unit_count', 'cas_id', 'fema_number', 'fema_flavor_profile', 'odor', 'taste', 'functional_groups', 'inchi', 'volume3d', 'fooddb_flavor_profile', 'super_sweet', 'bitter', 'supersweetdb_id', 'bitterdb_id', 'fooddb_id', 'flavornet_id', 'fenoroli_and_os', 'natural', 'unknown_natural', 'synthetic', 'flavor_profile']

Dataframe head:
   pubchem_id                          iupac_name  \
0           4                  1-aminopropan-2-ol   
1  

Encode Labels

In [31]:
df, label_encoder = encode_labels(df)

df[['flavor_profile', 'label']].head()

,flavor_profile,label
0,fishy,749
1,sweet,1860
3,fruity,838
4,odorless,1306
5,brown@caramel@lactonic@sweet@creamy,255


Select Features

In [32]:
X = df.select_dtypes(include=['float64', 'int64']).drop(columns=['label'], errors='ignore')
y = df['label']

print("Feature shape:", X.shape)

Feature shape: (25106, 29)


In [33]:
print("=" * 60)
print("DEBUGGING: Feature Selection")
print("=" * 60)
print(f"\n1. Columns in df: {df.columns.tolist()}")
print(f"\n2. Data types:\n{df.dtypes}")
print(f"\n3. Numeric columns (float64, int64):")
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(f"   {numeric_cols}")
print(f"\n4. Before dropping 'label': {len(numeric_cols)} columns")
if 'label' in numeric_cols:
    print(f"   Dropping 'label'...")
    numeric_cols.remove('label')
print(f"5. After dropping 'label': {len(numeric_cols)} columns")
print(f"\n6. X shape: {X.shape}")
print(f"7. y shape: {y.shape}")
print(f"8. X has NaN: {X.isnull().any().any()}")
print("=" * 60)

DEBUGGING: Feature Selection

1. Columns in df: ['pubchem_id', 'iupac_name', 'common_name', 'smile', 'molecular_weight', 'hbd_count', 'hba_count', 'num_rotatablebonds', 'complexity', 'topological_polor_surfacearea', 'monoisotopic_mass', 'exact_mass', 'xlogp', 'charge', 'heavy_atom_count', 'atom_stereo_count', 'defined_atom_stereocenter_count', 'undefined_atom_stereocenter_count', 'bond_stereo_count', 'defined_bond_stereocenter_count', 'undefined_bond_stereocenter_count', 'isotope_atom_count', 'covalently_bonded_unit_count', 'cas_id', 'fema_number', 'fema_flavor_profile', 'odor', 'taste', 'functional_groups', 'inchi', 'volume3d', 'fooddb_flavor_profile', 'super_sweet', 'bitter', 'supersweetdb_id', 'bitterdb_id', 'fooddb_id', 'flavornet_id', 'fenoroli_and_os', 'natural', 'unknown_natural', 'synthetic', 'flavor_profile', 'label']

2. Data types:
pubchem_id                             int64
iupac_name                               str
common_name                              str
smile     

Train Model (Problem 1)

In [34]:
print(f"✓ X shape: {X.shape}, y shape: {y.shape}")
print(f"✓ No NaN in X: {X.isnull().sum().sum() == 0}")
print(f"✓ Y value counts:\n{y.value_counts().head(10)}\n")

model = get_model()
model, X_test, y_test = train_model(model, X, y)

✓ X shape: (25106, 29), y shape: (25106,)
✓ No NaN in X: True
✓ Y value counts:
label
1861    13782
1860     8172
220       603
1306      120
838        31
233        12
1310       12
1307       12
29         10
768        10
Name: count, dtype: int64

Accuracy: 0.9044


Predict Probabilities

In [35]:
probs = model.predict_proba(X_test)

print("Sample probabilities:")
print(probs[:5])

Sample probabilities:
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


Explain Model (Problem 2)

In [1]:
explain_model(model, X_test)

NameError: name 'explain_model' is not defined